<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Baseline rule and reason codes

reason_codes = {
    "HIGH_OPPORTUNITY": "Stronger measured opportunity signal.",
    "MEDIUM_OPPORTUNITY": "Moderate opportunity signal.",
    "LOW_OPPORTUNITY": "Weaker opportunity signal.",
    "DATA_LIMIT": "Insufficient information for a confident recommendation."
}

print("Baseline rule: rank records using an observed-data opportunity score.")
print("\nReason codes:")
for code, description in reason_codes.items():
    print(f"{code}: {description}")


Baseline rule: rank records using an observed-data opportunity score.

Reason codes:
HIGH_OPPORTUNITY: Stronger measured opportunity signal.
MEDIUM_OPPORTUNITY: Moderate opportunity signal.
LOW_OPPORTUNITY: Weaker opportunity signal.
DATA_LIMIT: Insufficient information for a confident recommendation.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

repo_path = "/content/flyrank-ml-internship-starter"

if not os.path.exists(repo_path):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

print("Repository exists:", os.path.exists(repo_path))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 5.30 MiB/s, done.
Resolving deltas: 100% (161/161), done.
Repository exists: True


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

possible_paths = [
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (path for path in possible_paths if os.path.exists(path)),
    None
)

if csv_path is None:
    raise FileNotFoundError(
        "Starter CSV not found. Make sure the repository is available in Colab."
    )

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)



Dataset shape: (30000, 44)


In [6]:
# Fields that must not be used in the baseline score
excluded_fields = {
    "trend_direction",
    "trend_pct",
    "client_hash_id",
    "content_hash_id"
}

# Candidate numeric fields
numeric_cols = df.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

score_features = [
    col for col in numeric_cols
    if col not in excluded_fields
]

print("Numeric score candidates:")
print(score_features)

# Create a simple normalized score
score_parts = []

for col in score_features:
    series = pd.to_numeric(df[col], errors="coerce")

    if series.notna().sum() == 0:
        continue

    median = series.median()
    series = series.fillna(median)

    minimum = series.min()
    maximum = series.max()

    if maximum != minimum:
        normalized = (series - minimum) / (maximum - minimum)
        score_parts.append(normalized)

if not score_parts:
    raise ValueError("No usable numeric features were found for the baseline score.")

score = pd.concat(score_parts, axis=1).mean(axis=1)

baseline = pd.DataFrame({
    "row_id": range(len(df)),
    "action_score": score
})

baseline["rank"] = (
    baseline["action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Reason codes based on score distribution
q75 = baseline["action_score"].quantile(0.75)
q25 = baseline["action_score"].quantile(0.25)

baseline["reason_code"] = np.select(
    [
        baseline["action_score"] >= q75,
        baseline["action_score"] <= q25
    ],
    [
        "HIGH_OPPORTUNITY",
        "LOW_OPPORTUNITY"
    ],
    default="MEDIUM_OPPORTUNITY"
)

# Save required output
output_dir = "/content/flyrank-ml-internship-starter/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

baseline = baseline.sort_values("rank")

baseline.to_csv(output_path, index=False)

print("Rows ranked:", len(baseline))
print("Output:", output_path)
print("\nTop 5:")
display(baseline.head())

Numeric score candidates:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Rows ranked: 30000
Output: /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv

Top 5:


,row_id,action_score,rank,reason_code
29400,29400,0.423575,1,HIGH_OPPORTUNITY
10741,10741,0.414457,2,HIGH_OPPORTUNITY
21565,21565,0.365559,3,HIGH_OPPORTUNITY
13537,13537,0.338706,4,HIGH_OPPORTUNITY
16811,16811,0.335106,5,HIGH_OPPORTUNITY


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Top-20 review

top20 = baseline.head(20).copy()

top20["action"] = "Review for content refresh"

top20["confidence_note"] = top20["reason_code"].map({
    "HIGH_OPPORTUNITY": "Higher relative baseline score.",
    "MEDIUM_OPPORTUNITY": "Moderate relative baseline score.",
    "LOW_OPPORTUNITY": "Lower relative baseline score.",
    "DATA_LIMIT": "Insufficient information for a confident recommendation."
}).fillna("Limited signal.")

top20["what_would_make_it_wrong"] = (
    "The score may reflect noisy, incomplete, or non-actionable information."
)

review_columns = [
    "rank",
    "row_id",
    "action",
    "reason_code",
    "action_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("Top-20 review:")
display(top20[review_columns])

Top-20 review:


,rank,row_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
29400,1,29400,Review for content refresh,HIGH_OPPORTUNITY,0.423575,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
10741,2,10741,Review for content refresh,HIGH_OPPORTUNITY,0.414457,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
21565,3,21565,Review for content refresh,HIGH_OPPORTUNITY,0.365559,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
13537,4,13537,Review for content refresh,HIGH_OPPORTUNITY,0.338706,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
16811,5,16811,Review for content refresh,HIGH_OPPORTUNITY,0.335106,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
17812,6,17812,Review for content refresh,HIGH_OPPORTUNITY,0.332228,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
21819,7,21819,Review for content refresh,HIGH_OPPORTUNITY,0.318423,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
14178,8,14178,Review for content refresh,HIGH_OPPORTUNITY,0.318262,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
14234,9,14234,Review for content refresh,HIGH_OPPORTUNITY,0.298699,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."
2797,10,2797,Review for content refresh,HIGH_OPPORTUNITY,0.297626,Higher relative baseline score.,"The score may reflect noisy, incomplete, or no..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 4. Weak picks + leakage check

print("Lowest 5 baseline records:")
display(baseline.tail(5))

# Known label-derived fields
label_derived_fields = [
    "trend_direction",
    "trend_pct"
]

# Search for possible future-window fields
future_keywords = [
    "future",
    "post",
    "after",
    "next",
    "later"
]

# Search for possible product-related flags
product_keywords = [
    "product",
    "plan",
    "package",
    "tier"
]

# Fields actually used by the baseline score
used_fields = set(score_features)

label_leakage = [
    col for col in label_derived_fields
    if col in used_fields
]

future_fields = [
    col for col in used_fields
    if any(word in col.lower() for word in future_keywords)
]

product_fields = [
    col for col in used_fields
    if any(word in col.lower() for word in product_keywords)
]

print("\nLabel-derived fields used:")
print(label_leakage)

print("\nPossible future-window fields used:")
print(future_fields)

print("\nPossible product-related fields used:")
print(product_fields)

# Fail the notebook if known label leakage is detected
assert not label_leakage, (
    f"Known label leakage detected: {label_leakage}"
)

print("\nKnown label leakage check: PASSED")

Lowest 5 baseline records:


,row_id,action_score,rank,reason_code
13188,13188,0.018281,29996,LOW_OPPORTUNITY
20730,20730,0.018262,29997,LOW_OPPORTUNITY
15767,15767,0.018255,29998,LOW_OPPORTUNITY
29455,29455,0.016846,29999,LOW_OPPORTUNITY
15027,15027,0.016173,30000,LOW_OPPORTUNITY



Label-derived fields used:
[]

Possible future-window fields used:
[]

Possible product-related fields used:
['age_tier_order']

Known label leakage check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.